# Multi-Agent LLM Framework — Experimento BPI Challenge 2019
## Validación sobre Benchmark Externo — PROJENER.AI SL · UAX · Máster Big Data · 2026

**Autora:** Edurne Martínez de Contrasta  
**Framework:** CrewAI · Groq API · Python 3.12  
**Modelos:** llama-3.1-8b-instant · llama-3.3-70b-versatile  

---

## Propósito del experimento

Este notebook replica el experimento principal (RPA-Baseline, Single-Small, Multi-Small, Multi-Mixed) sobre el BPI Challenge 2019, un dataset real de procurement de una empresa multinacional holandesa con ~251.734 casos.

**Fuente:** BPI Challenge 2019, Boudewijn van Dongen, TU/e  
**DOI:** 10.4121/uuid:d06aff4b-79f0-45e6-8ec8-e19730c248f1  
**Licencia:** CC BY 4.0

**Diferencias respecto al experimento principal:**
- **Dataset:** 50 casos muestreados del BPI real (vs. 50 casos sintéticos)
- **Proveedores:** IDs reales anonimizados del BPI (vendorID_XXXX)
- **Umbrales financieros:** basados en estadísticas reales del dataset
- **Categorías:** 3-way match, Consignment, 2-way match del BPI
- **Ground truth:** derivado de señales objetivas del event log real

**Métricas:**
- **ARR** (Autonomous Resolution Rate): % casos resueltos sin HiL. Objetivo: >70%
- **HDA** (HiL Detection Accuracy): % casos HiL detectados correctamente. Métrica exploratoria (n=11 casos positivos)
- **DER** (Decision Error Rate): % decisiones incorrectas. Objetivo: <10%

In [1]:
import os
os.makedirs(r'C:\Users\edurn\practicas projener\resultados', exist_ok=True)

## 1. Setup y verificación del entorno

In [4]:
import sys, os
print(sys.version)

os.chdir(r'C:\Users\edurn\practicas projener')
sys.path.insert(0, r'C:\Users\edurn\practicas projener')

archivos_necesarios = [
    'api_proveedores_bpi.py',
    'api_finance_bpi.py',
    'api_compliance_bpi.py',
    'procurement_apis_bpi.py',
    'casos_bpi_adaptados.json',
    'api_legal.py',
    'api_requester.py',
    'api_hil.py'
]

print('\nDirectorio actual:', os.getcwd())
print()
for archivo in archivos_necesarios:
    existe = os.path.exists(archivo)
    estado = '✅' if existe else '❌ NO ENCONTRADO'
    print(f'{estado} — {archivo}')

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]

Directorio actual: C:\Users\edurn\practicas projener

✅ — api_proveedores_bpi.py
✅ — api_finance_bpi.py
✅ — api_compliance_bpi.py
✅ — procurement_apis_bpi.py
✅ — casos_bpi_adaptados.json
✅ — api_legal.py
✅ — api_requester.py
✅ — api_hil.py


In [5]:
from procurement_apis_bpi import (
    get_proveedor, verificar_presupuesto, verificar_normativa,
    cargar_casos_bpi, get_estadisticas_dataset
)
from api_legal import validar_contrato
from api_hil import escalar_a_humano

print('=' * 60)
print('TEST APIs BPI Challenge 2019')
print('=' * 60)

stats = get_estadisticas_dataset()
print(f'\nFuente: {stats["fuente"]}')
print(f'DOI: {stats["doi"]}')
print(f'Total casos BPI: {stats["total_casos_bpi"]:,}')
print(f'Casos evaluación: {stats["casos_evaluacion"]}')
print(f'Distribución: {stats["distribucion"]}')
print(f'Casos HiL: {stats["casos_hil"]}')
print(f'Importe medio real: {stats["importe_medio"]:.0f}€')
print(f'Importe máximo real: {stats["importe_max"]:,.0f}€')

r = get_proveedor('vendorID_0226')
print(f'\nProveedor BPI vendorID_0226: {r["estado"]} | opera: {r["puede_operar"]}')

r = verificar_presupuesto('Operations', 3683.0, 'Standard PO')
print(f'Presupuesto 3683€ Standard PO: aprobado={r["aprobado"]} nivel={r["nivel_aprobacion_requerido"]}')

r = verificar_normativa('Consignment', True, [], 0)
print(f'Normativa Consignment: cumple={r["cumple_normativa"]} riesgo={r["nivel_riesgo"]} HiL={r["requiere_escalacion_hil"]}')

print('\n✅ APIs BPI funcionan correctamente.')

TEST APIs BPI Challenge 2019

Fuente: BPI Challenge 2019
DOI: 10.4121/uuid:d06aff4b-79f0-45e6-8ec8-e19730c248f1
Total casos BPI: 251,734
Casos evaluación: 50
Distribución: {'Simple': 20, 'Medio': 18, 'Complejo': 12}
Casos HiL: 11
Importe medio real: 12711€
Importe máximo real: 372,741€

Proveedor BPI vendorID_0226: aprobado | opera: True
Presupuesto 3683€ Standard PO: aprobado=True nivel=manager
Normativa Consignment: cumple=False riesgo=alto HiL=True

✅ APIs BPI funcionan correctamente.


## 2. Dataset BPI Challenge 2019 — 50 casos reales

50 casos muestreados del BPI Challenge 2019 con ground truth derivado del event log real:
- **Simple** (20 casos): Standard PO, importes bajos, proceso limpio
- **Medio** (18 casos): importes medios, algunos invoice after GR
- **Complejo** (12 casos): Consignment, Framework order >372k€, cancelaciones

De los 50 casos, **11 requieren escalación HiL** según ground truth derivado del BPI.

In [6]:
import json

with open('casos_bpi_adaptados.json', encoding='utf-8') as f:
    data = json.load(f)
casos_bpi = data['casos']

print(f'Casos cargados: {len(casos_bpi)}')
print()

niveles = {}
for c in casos_bpi:
    niveles[c['nivel']] = niveles.get(c['nivel'], 0) + 1
for nivel, n in niveles.items():
    print(f'  {nivel}: {n} casos')

hil_total = sum(1 for c in casos_bpi if c['ground_truth']['requiere_hil'])
print(f'\nCasos que requieren HiL: {hil_total}')
print(f'Casos sin HiL: {len(casos_bpi) - hil_total}')

print('\nEjemplo caso Simple BPI:')
simple = [c for c in casos_bpi if c['nivel'] == 'Simple'][0]
print(f'  ID: {simple["id"]} | Caso real BPI: {simple["caso_real_bpi"]}')
print(f'  Importe: {simple["importe"]}€ | Tipo: {simple["tipo_documento_bpi"]}')
print(f'  Categoría BPI: {simple["categoria_bpi"]}')
print(f'  Actividades: {simple["actividades_reales"]}')

print('\nEjemplo caso Complejo BPI:')
complejo = [c for c in casos_bpi if c['ground_truth']['requiere_hil']][0]
print(f'  ID: {complejo["id"]} | Caso real BPI: {complejo["caso_real_bpi"]}')
print(f'  Importe: {complejo["importe"]}€ | Tipo: {complejo["tipo_documento_bpi"]}')
print(f'  Categoría BPI: {complejo["categoria_bpi"]}')
print(f'  Actividades: {complejo["actividades_reales"]}')
print(f'  Ground truth HiL: {complejo["ground_truth"]["requiere_hil"]}')

Casos cargados: 50

  Simple: 20 casos
  Medio: 18 casos
  Complejo: 12 casos

Casos que requieren HiL: 11
Casos sin HiL: 39

Ejemplo caso Simple BPI:
  ID: BPI_01 | Caso real BPI: 4507032049_00010
  Importe: 3683.0€ | Tipo: Standard PO
  Categoría BPI: 3-way match, invoice before GR
  Actividades: ['Create Purchase Order Item', 'Vendor creates invoice', 'Record Goods Receipt', 'Record Invoice Receipt', 'Clear Invoice']

Ejemplo caso Complejo BPI:
  ID: BPI_39 | Caso real BPI: 4507040511_00010
  Importe: 0.0€ | Tipo: Standard PO
  Categoría BPI: Consignment
  Actividades: ['Create Purchase Order Item', 'Change Quantity', 'Record Goods Receipt']
  Ground truth HiL: True


## 3. RPA-Baseline sobre BPI Challenge 2019

In [7]:
import time

def m1_bpi_procesar_caso(caso):
    inicio = time.time()
    decision = None
    escalo_hil = False

    # REGLA 1: Consignment — RPA aprueba sin entender el contrato marco
    if caso['categoria_bpi'] == 'Consignment':
        decision = 'aprobado'
        razon = 'Consignment: RPA aprueba por importe=0 sin evaluar contrato marco'

    # REGLA 2: Framework order >100k — RPA no tiene umbral para esto
    elif caso['tipo_documento_bpi'] == 'Framework order' and caso['importe'] > 100000:
        decision = 'aprobado'
        razon = 'Framework order: RPA aprueba sin verificar nivel de aprobación requerido'

    # REGLA 3: importe <= umbral automático BPI (P75 real = 2109€)
    elif caso['importe'] <= 2109:
        decision = 'aprobado'
        razon = 'Importe dentro del límite automático BPI (≤2109€ P75 real)'

    # REGLA 4: verificar proveedor si importe > 2109€
    else:
        r_prov = get_proveedor(caso['proveedor_id'])
        if not r_prov.get('puede_operar', False):
            decision = 'rechazado'
            razon = f'Proveedor {caso["proveedor_id"]} no puede operar'
        else:
            r_fin = verificar_presupuesto(
                caso['departamento'], caso['importe'], caso['tipo_documento_bpi']
            )
            if not r_fin['aprobado']:
                decision = 'rechazado'
                razon = 'Presupuesto insuficiente'
            else:
                decision = 'aprobado'
                razon = 'Proveedor OK + presupuesto OK'

    tiempo = round(time.time() - inicio, 4)
    return {
        'caso_id': caso['id'],
        'caso_real_bpi': caso['caso_real_bpi'],
        'nivel': caso['nivel'],
        'importe': caso['importe'],
        'categoria_bpi': caso['categoria_bpi'],
        'decision_m1': decision,
        'razon': razon,
        'escalo_hil': escalo_hil,
        'tiempo_seg': tiempo,
        'tokens_usados': 0,
        'ground_truth': caso['ground_truth']
    }

def calcular_metricas(resultados, modelo_key, modelo_nombre):
    total = len(resultados)
    resueltos = sum(1 for r in resultados if not r['escalo_hil'] and r[modelo_key] != 'error')
    deben_hil = [r for r in resultados if r['ground_truth']['requiere_hil']]
    hil_ok = sum(1 for r in deben_hil if r['escalo_hil'])
    errores = sum(1 for r in resultados if
        (r['ground_truth']['requiere_hil'] and not r['escalo_hil']) or
        (r[modelo_key] == 'rechazado' and 'aprobacion' in r['ground_truth']['resultado'])
    )
    return {
        'modelo': modelo_nombre,
        'dataset': 'BPI_Challenge_2019',
        'total_casos': total,
        'ARR': round(resueltos / total * 100, 1),
        'HDA': round(hil_ok / len(deben_hil) * 100, 1) if deben_hil else 0,
        'DER': round(errores / total * 100, 1),
        'PT': round(sum(r['tiempo_seg'] for r in resultados) / total, 2),
        'TCC': 0
    }

print('Ejecutando M1 sobre 50 casos BPI Challenge 2019...\n')
resultados_m1_bpi = []

for caso in casos_bpi:
    resultado = m1_bpi_procesar_caso(caso)
    resultados_m1_bpi.append(resultado)
    estado = '🔶 HiL' if resultado['escalo_hil'] else '✅'
    print(f"  {caso['id']} [{caso['nivel']:8}] → {resultado['decision_m1']:15} {estado}")

metricas_m1_bpi = calcular_metricas(resultados_m1_bpi, 'decision_m1', 'M1_RPA_BPI2019')

print('\n' + '=' * 55)
print(f"  RESULTADOS M1 — BPI Challenge 2019")
print('=' * 55)
print(f"  ARR: {metricas_m1_bpi['ARR']}%   (objetivo >70%)")
print(f"  HDA: {metricas_m1_bpi['HDA']}%   (objetivo >90%)")
print(f"  DER: {metricas_m1_bpi['DER']}%   (objetivo <10%)")
print(f"  PT:  {metricas_m1_bpi['PT']} seg/caso")
print('=' * 55)

with open('resultados/resultados_m1_bpi.json', 'w', encoding='utf-8') as f:
    json.dump({'modelo': 'M1_BPI', 'metricas': metricas_m1_bpi,
               'resultados_por_caso': resultados_m1_bpi}, f, ensure_ascii=False, indent=2)
print('\n✅ M1 BPI completado y guardado.')

Ejecutando M1 sobre 50 casos BPI Challenge 2019...

  BPI_01 [Simple  ] → aprobado        ✅
  BPI_02 [Simple  ] → aprobado        ✅
  BPI_03 [Simple  ] → aprobado        ✅
  BPI_04 [Simple  ] → aprobado        ✅
  BPI_05 [Simple  ] → aprobado        ✅
  BPI_06 [Simple  ] → aprobado        ✅
  BPI_07 [Simple  ] → aprobado        ✅
  BPI_08 [Simple  ] → aprobado        ✅
  BPI_09 [Simple  ] → aprobado        ✅
  BPI_10 [Simple  ] → aprobado        ✅
  BPI_11 [Simple  ] → aprobado        ✅
  BPI_12 [Simple  ] → aprobado        ✅
  BPI_13 [Simple  ] → aprobado        ✅
  BPI_14 [Simple  ] → aprobado        ✅
  BPI_15 [Simple  ] → aprobado        ✅
  BPI_16 [Simple  ] → aprobado        ✅
  BPI_17 [Simple  ] → aprobado        ✅
  BPI_18 [Simple  ] → aprobado        ✅
  BPI_19 [Simple  ] → aprobado        ✅
  BPI_20 [Simple  ] → aprobado        ✅
  BPI_21 [Medio   ] → aprobado        ✅
  BPI_22 [Medio   ] → aprobado        ✅
  BPI_23 [Medio   ] → aprobado        ✅
  BPI_24 [Medio   ] → aproba

In [2]:
import sys
import os
sys.path.insert(0, r'C:\Users\edurn\venv_practicas\Lib\site-packages')
os.chdir(r'C:\Users\edurn\practicas projener')
sys.path.insert(0, r'C:\Users\edurn\practicas projener')

import crewai
print(f'CrewAI version: {crewai.__version__}')

CrewAI version: 1.14.2


## 4. Single-Small sobre BPI Challenge 2019

In [8]:
import re
os.environ["GROQ_API_KEY"] = "TU_CLAVE_GROQ_AQUI"  # Obtén tu clave en https://console.groq.com
os.environ['OTEL_SDK_DISABLED'] = 'true'

from crewai import Agent, Task, Crew

MODELO_8B = 'groq/llama-3.1-8b-instant'

def procesar_m2_bpi(caso):
    inicio = time.time()

    r_prov = get_proveedor(caso['proveedor_id'])
    r_fin  = verificar_presupuesto(
        caso['departamento'], caso['importe'], caso['tipo_documento_bpi']
    )
    r_comp = verificar_normativa(
        caso['categoria_bpi'], caso['goods_receipt'],
        caso['actividades_reales'], caso['importe']
    )

    # Contexto enriquecido con datos reales del BPI
    ctx = (
        f"ID:{caso['id']} CasoRealBPI:{caso['caso_real_bpi']} "
        f"{caso['descripcion'][:60]} "
        f"Importe:{caso['importe']}EUR "
        f"TipoDoc:{caso['tipo_documento_bpi']} "
        f"CategoriaBPI:{caso['categoria_bpi']} "
        f"Proveedor:{r_prov.get('estado','?')} opera={r_prov.get('puede_operar','?')} "
        f"Presupuesto:ok={r_fin.get('aprobado','?')} nivel={r_fin.get('nivel_aprobacion_requerido','?')} "
        f"Compliance:cumple={r_comp.get('cumple_normativa','?')} riesgo={r_comp.get('nivel_riesgo','?')} "
        f"HiLrequerido={r_comp.get('requiere_escalacion_hil','?')} "
        f"Actividades:{caso['actividades_reales'][:3]}"
    )

    a = Agent(
        role='Gestor Compras BPI',
        goal='Decidir sobre solicitudes de compra reales del BPI Challenge 2019.',
        backstory='Experto en procurement con acceso a datos reales del BPI 2019.',
        verbose=False, allow_delegation=False, llm=MODELO_8B
    )
    t = Task(
        description='Analiza y responde SOLO JSON: {"decision":"aprobado o rechazado o escalado_hil","razon":"frase corta","escala_hil":false}\n\n' + ctx,
        expected_output='JSON',
        agent=a
    )
    crew = Crew(agents=[a], tasks=[t], verbose=False)

    try:
        txt = str(crew.kickoff()).strip()
        m = re.search(r'\{.*?\}', txt, re.DOTALL)
        if m:
            d = json.loads(m.group())
            dec = d.get('decision', 'rechazado')
            raz = d.get('razon', '')
            esc = d.get('escala_hil', False)
        else:
            dec, raz, esc = 'rechazado', 'no parse', False
    except Exception as e:
        dec, raz, esc = 'error', str(e)[:60], False

    return {
        'caso_id': caso['id'],
        'caso_real_bpi': caso['caso_real_bpi'],
        'nivel': caso['nivel'],
        'importe': caso['importe'],
        'categoria_bpi': caso['categoria_bpi'],
        'decision_m2': dec,
        'razon': raz,
        'escalo_hil': esc,
        'tiempo_seg': round(time.time() - inicio, 2),
        'tokens_usados': 200,
        'ground_truth': caso['ground_truth']
    }

print('Ejecutando M2 sobre 50 casos BPI Challenge 2019...\n')
resultados_m2_bpi = []

for caso in casos_bpi:
    print(f"  {caso['id']} [{caso['nivel']:8}]...", end=' ')
    intentos = 0
    r = None
    while intentos < 2:
        r = procesar_m2_bpi(caso)
        if r['decision_m2'] != 'error':
            break
        intentos += 1
        print('(reintento)...', end=' ')
        time.sleep(3)
    resultados_m2_bpi.append(r)
    estado = '🔶' if r['escalo_hil'] else '✅'
    if r['decision_m2'] == 'error':
        estado = '❌'
    print(f"→ {r['decision_m2']} {estado} ({r['tiempo_seg']}s)")
    time.sleep(4)

metricas_m2_bpi = calcular_metricas(resultados_m2_bpi, 'decision_m2', 'M2_SingleAgent_BPI2019')

print('\n' + '=' * 55)
print('  RESULTADOS M2 — BPI Challenge 2019')
print('=' * 55)
print(f"  ARR: {metricas_m2_bpi['ARR']}%   (objetivo >70%)")
print(f"  HDA: {metricas_m2_bpi['HDA']}%   (objetivo >90%)")
print(f"  DER: {metricas_m2_bpi['DER']}%   (objetivo <10%)")
print(f"  PT:  {metricas_m2_bpi['PT']} seg/caso")
print('=' * 55)

with open('resultados/resultados_m2_bpi.json', 'w', encoding='utf-8') as f:
    json.dump({'modelo': 'M2_BPI', 'metricas': metricas_m2_bpi,
               'resultados_por_caso': resultados_m2_bpi}, f, ensure_ascii=False, indent=2)
print('\n✅ M2 BPI completado y guardado.')

Ejecutando M2 sobre 50 casos BPI Challenge 2019...

  BPI_01 [Simple  ]... → aprobado ✅ (0.66s)
  BPI_02 [Simple  ]... → aprobado ✅ (0.31s)
  BPI_03 [Simple  ]... → aprobado ✅ (1.24s)
  BPI_04 [Simple  ]... → aprobado ✅ (0.5s)
  BPI_05 [Simple  ]... → aprobado ✅ (0.4s)
  BPI_06 [Simple  ]... → aprobado ✅ (0.4s)
  BPI_07 [Simple  ]... → rechazado 🔶 (0.42s)
  BPI_08 [Simple  ]... → aprobado ✅ (0.48s)
  BPI_09 [Simple  ]... → rechazado ✅ (0.41s)
  BPI_10 [Simple  ]... → aprobado ✅ (0.5s)
  BPI_11 [Simple  ]... → aprobado ✅ (0.49s)
  BPI_12 [Simple  ]... → aprobado ✅ (0.42s)
  BPI_13 [Simple  ]... → aprobado ✅ (0.48s)
  BPI_14 [Simple  ]... → aprobado ✅ (0.39s)
  BPI_15 [Simple  ]... → aprobado ✅ (0.45s)
  BPI_16 [Simple  ]... → escalado_hil 🔶 (0.45s)
  BPI_17 [Simple  ]... → aprobado ✅ (0.36s)
  BPI_18 [Simple  ]... → aprobado ✅ (0.39s)
  BPI_19 [Simple  ]... → aprobado ✅ (0.58s)
  BPI_20 [Simple  ]... → aprobado ✅ (0.38s)
  BPI_21 [Medio   ]... → aprobado ✅ (0.52s)
  BPI_22 [Medio   ]...

## 5. Multi-Small sobre BPI Challenge 2019

In [9]:
def procesar_m3_bpi(caso):
    inicio = time.time()

    prov  = get_proveedor(caso['proveedor_id'])
    fin   = verificar_presupuesto(
        caso['departamento'], caso['importe'], caso['tipo_documento_bpi']
    )
    comp  = verificar_normativa(
        caso['categoria_bpi'], caso['goods_receipt'],
        caso['actividades_reales'], caso['importe']
    )
    legal = validar_contrato(
        caso['categoria_compra'], caso.get('clausulas', []), caso['importe'], 12
    )

    prov_txt  = f"estado={prov.get('estado','?')} opera={prov.get('puede_operar','?')}"
    fin_txt   = f"aprobado={fin.get('aprobado','?')} nivel={fin.get('nivel_aprobacion_requerido','?')}"
    comp_txt  = f"cumple={comp.get('cumple_normativa','?')} riesgo={comp.get('nivel_riesgo','?')} hil={comp.get('requiere_escalacion_hil','?')}"
    legal_txt = f"riesgo={legal.get('nivel_riesgo_global','?')}"
    bpi_ctx   = f"TipoDoc:{caso['tipo_documento_bpi']} CategoriaBPI:{caso['categoria_bpi']} Actividades:{caso['actividades_reales'][:3]}"

    a1 = Agent(role='Requester', goal='Registrar solicitud BPI.',
               backstory='Registras solicitudes reales del BPI 2019.',
               verbose=False, allow_delegation=False, llm=MODELO_8B)
    a2 = Agent(role='Procurement', goal='Validar proveedor BPI.',
               backstory=f'Proveedor BPI: {prov_txt}',
               verbose=False, allow_delegation=False, llm=MODELO_8B)
    a3 = Agent(role='Finance', goal='Verificar presupuesto BPI.',
               backstory=f'Finanzas BPI: {fin_txt}',
               verbose=False, allow_delegation=False, llm=MODELO_8B)
    a4 = Agent(role='Legal', goal='Revisar contrato BPI.',
               backstory=f'Legal BPI: {legal_txt}',
               verbose=False, allow_delegation=False, llm=MODELO_8B)
    a5 = Agent(role='Compliance', goal='Decidir cumplimiento BPI.',
               backstory=f'Normativa BPI: {comp_txt} | {bpi_ctx}',
               verbose=False, allow_delegation=False, llm=MODELO_8B)

    t1 = Task(description=f"Solicitud {caso['id']} {caso['importe']}EUR BPI. Confirma en UNA frase.",
              expected_output='OK.', agent=a1)
    t2 = Task(description=f"Proveedor {caso['proveedor_id']} BPI. Datos en backstory. UNA frase.",
              expected_output='OK.', agent=a2)
    t3 = Task(description=f"Presupuesto {caso['departamento']} {caso['importe']}EUR BPI. UNA frase.",
              expected_output='OK.', agent=a3)
    t4 = Task(description=f"Contrato {caso['categoria_bpi']} BPI. Datos en backstory. UNA frase.",
              expected_output='OK.', agent=a4)
    t5 = Task(description='Decide SOLO JSON: {"decision":"aprobado o rechazado o escalado_hil","razon":"frase","escala_hil":false}',
              expected_output='JSON.', agent=a5)

    crew = Crew(agents=[a1,a2,a3,a4,a5], tasks=[t1,t2,t3,t4,t5], verbose=False)

    try:
        txt = str(crew.kickoff()).strip()
        m = re.search(r'\{.*?\}', txt, re.DOTALL)
        if m:
            d = json.loads(m.group())
            dec = d.get('decision', 'rechazado')
            raz = d.get('razon', '')
            esc = d.get('escala_hil', False)
        else:
            dec, raz, esc = 'rechazado', 'no parse', False
    except Exception as e:
        dec, raz, esc = 'error', str(e)[:60], False

    return {
        'caso_id': caso['id'],
        'caso_real_bpi': caso['caso_real_bpi'],
        'nivel': caso['nivel'],
        'importe': caso['importe'],
        'categoria_bpi': caso['categoria_bpi'],
        'decision_m3': dec,
        'razon': raz,
        'escalo_hil': esc,
        'tiempo_seg': round(time.time() - inicio, 2),
        'tokens_usados': 750,
        'ground_truth': caso['ground_truth']
    }

print('Ejecutando M3 sobre 50 casos BPI Challenge 2019...\n')
resultados_m3_bpi = []

for caso in casos_bpi:
    print(f"  {caso['id']} [{caso['nivel']:8}]...", end=' ')
    intentos = 0
    r = None
    while intentos < 3:
        r = procesar_m3_bpi(caso)
        if r['decision_m3'] != 'error':
            break
        intentos += 1
        print(f'(reintento {intentos})...', end=' ')
        time.sleep(10)
    resultados_m3_bpi.append(r)
    estado = '🔶' if r['escalo_hil'] else '✅'
    if r['decision_m3'] == 'error':
        estado = '❌'
    print(f"→ {r['decision_m3']} {estado} ({r['tiempo_seg']}s)")
    time.sleep(5)

metricas_m3_bpi = calcular_metricas(resultados_m3_bpi, 'decision_m3', 'M3_MultiAgent_BPI2019')

print('\n' + '=' * 55)
print('  RESULTADOS M3 — BPI Challenge 2019')
print('=' * 55)
print(f"  ARR: {metricas_m3_bpi['ARR']}%   (objetivo >70%)")
print(f"  HDA: {metricas_m3_bpi['HDA']}%   (objetivo >90%)")
print(f"  DER: {metricas_m3_bpi['DER']}%   (objetivo <10%)")
print(f"  PT:  {metricas_m3_bpi['PT']} seg/caso")
print('=' * 55)

with open('resultados/resultados_m3_bpi.json', 'w', encoding='utf-8') as f:
    json.dump({'modelo': 'M3_BPI', 'metricas': metricas_m3_bpi,
               'resultados_por_caso': resultados_m3_bpi}, f, ensure_ascii=False, indent=2)
print('\n✅ M3 BPI completado y guardado.')

Ejecutando M3 sobre 50 casos BPI Challenge 2019...

  BPI_01 [Simple  ]... → rechazado ✅ (1.72s)
  BPI_02 [Simple  ]... → aprobado ✅ (2.16s)
  BPI_03 [Simple  ]... → aprobado ✅ (1.85s)
  BPI_04 [Simple  ]... → aprobado ✅ (1.97s)
  BPI_05 [Simple  ]... → aprobado ✅ (2.88s)
  BPI_06 [Simple  ]... → aprobado ✅ (2.06s)
  BPI_07 [Simple  ]... → rechazado ✅ (2.37s)
  BPI_08 [Simple  ]... → aprobado ✅ (1.86s)
  BPI_09 [Simple  ]... → aprobado ✅ (1.65s)
  BPI_10 [Simple  ]... (reintento 1)... → aprobado ✅ (2.13s)
  BPI_11 [Simple  ]... (reintento 1)... (reintento 2)... (reintento 3)... → error ❌ (2.01s)
  BPI_12 [Simple  ]... → aprobado ✅ (1.86s)
  BPI_13 [Simple  ]... (reintento 1)... → rechazado ✅ (1.76s)
  BPI_14 [Simple  ]... (reintento 1)... (reintento 2)... (reintento 3)... → error ❌ (1.57s)
  BPI_15 [Simple  ]... (reintento 1)... → escalado_hil ✅ (2.08s)
  BPI_16 [Simple  ]... (reintento 1)... → rechazado ✅ (1.76s)
  BPI_17 [Simple  ]... (reintento 1)... → aprobado ✅ (2.19s)
  BPI_18 [S

## 6. Multi-Mixed sobre BPI Challenge 2019

In [10]:
MODELO_70B = 'groq/llama-3.3-70b-versatile'

def procesar_m4_bpi(caso):
    inicio = time.time()

    prov  = get_proveedor(caso['proveedor_id'])
    fin   = verificar_presupuesto(
        caso['departamento'], caso['importe'], caso['tipo_documento_bpi']
    )
    comp  = verificar_normativa(
        caso['categoria_bpi'], caso['goods_receipt'],
        caso['actividades_reales'], caso['importe']
    )
    legal = validar_contrato(
        caso['categoria_compra'], caso.get('clausulas', []), caso['importe'], 12
    )

    prov_txt  = f"estado={prov.get('estado','?')} opera={prov.get('puede_operar','?')}"
    fin_txt   = f"aprobado={fin.get('aprobado','?')} nivel={fin.get('nivel_aprobacion_requerido','?')}"
    comp_txt  = f"cumple={comp.get('cumple_normativa','?')} riesgo={comp.get('nivel_riesgo','?')} hil={comp.get('requiere_escalacion_hil','?')}"
    legal_txt = f"riesgo={legal.get('nivel_riesgo_global','?')}"
    bpi_ctx   = f"TipoDoc:{caso['tipo_documento_bpi']} CategoriaBPI:{caso['categoria_bpi']} Actividades:{caso['actividades_reales']}"

    # Agentes auxiliares con 8b, decisor con 70b
    a1 = Agent(role='Requester', goal='Registrar solicitud BPI.',
               backstory='Registras solicitudes reales del BPI 2019.',
               verbose=False, allow_delegation=False, llm=MODELO_8B)
    a2 = Agent(role='Procurement', goal='Validar proveedor BPI.',
               backstory=f'Proveedor BPI: {prov_txt}',
               verbose=False, allow_delegation=False, llm=MODELO_8B)
    a3 = Agent(role='Finance', goal='Verificar presupuesto BPI.',
               backstory=f'Finanzas BPI: {fin_txt}',
               verbose=False, allow_delegation=False, llm=MODELO_8B)
    a4 = Agent(role='Legal', goal='Revisar contrato BPI.',
               backstory=f'Legal BPI: {legal_txt}',
               verbose=False, allow_delegation=False, llm=MODELO_8B)
    # Decisor con 70b — modelo potente para la decisión final
    a5 = Agent(role='Compliance', goal='Decidir cumplimiento BPI con criterio experto.',
               backstory=(
                   f'Normativa BPI: {comp_txt} | {bpi_ctx} | '
                   f'ESCALA si: Consignment sin contrato marco, importe >100000EUR, '
                   f'Framework order estratégico, proveedor con alertas, '
                   f'múltiples cancelaciones en el proceso real.'
               ),
               verbose=False, allow_delegation=False, llm=MODELO_70B)

    t1 = Task(description=f"Solicitud {caso['id']} {caso['importe']}EUR BPI. Confirma en UNA frase.",
              expected_output='OK.', agent=a1)
    t2 = Task(description=f"Proveedor {caso['proveedor_id']} BPI. Datos en backstory. UNA frase.",
              expected_output='OK.', agent=a2)
    t3 = Task(description=f"Presupuesto {caso['departamento']} {caso['importe']}EUR BPI. UNA frase.",
              expected_output='OK.', agent=a3)
    t4 = Task(description=f"Contrato {caso['categoria_bpi']} BPI. Datos en backstory. UNA frase.",
              expected_output='OK.', agent=a4)
    t5 = Task(description='Decide SOLO JSON: {"decision":"aprobado o rechazado o escalado_hil","razon":"frase","escala_hil":false}',
              expected_output='JSON.', agent=a5)

    crew = Crew(agents=[a1,a2,a3,a4,a5], tasks=[t1,t2,t3,t4,t5], verbose=False)

    try:
        txt = str(crew.kickoff()).strip()
        m = re.search(r'\{.*?\}', txt, re.DOTALL)
        if m:
            d = json.loads(m.group())
            dec = d.get('decision', 'rechazado')
            raz = d.get('razon', '')
            esc = d.get('escala_hil', False)
        else:
            dec, raz, esc = 'rechazado', 'no parse', False
    except Exception as e:
        dec, raz, esc = 'error', str(e)[:60], False

    return {
        'caso_id': caso['id'],
        'caso_real_bpi': caso['caso_real_bpi'],
        'nivel': caso['nivel'],
        'importe': caso['importe'],
        'categoria_bpi': caso['categoria_bpi'],
        'decision_m4': dec,
        'razon': raz,
        'escalo_hil': esc,
        'tiempo_seg': round(time.time() - inicio, 2),
        'tokens_usados': 750,
        'ground_truth': caso['ground_truth']
    }

print('Ejecutando M4 sobre 50 casos BPI Challenge 2019...\n')
resultados_m4_bpi = []

for caso in casos_bpi:
    print(f"  {caso['id']} [{caso['nivel']:8}]...", end=' ')
    intentos = 0
    r = None
    while intentos < 3:
        r = procesar_m4_bpi(caso)
        if r['decision_m4'] != 'error':
            break
        intentos += 1
        print(f'(reintento {intentos})...', end=' ')
        time.sleep(10)
    resultados_m4_bpi.append(r)
    estado = '🔶' if r['escalo_hil'] else '✅'
    if r['decision_m4'] == 'error':
        estado = '❌'
    print(f"→ {r['decision_m4']} {estado} ({r['tiempo_seg']}s)")
    time.sleep(8)

metricas_m4_bpi = calcular_metricas(resultados_m4_bpi, 'decision_m4', 'M4_MultiMixto_BPI2019')

print('\n' + '=' * 55)
print('  RESULTADOS M4 — BPI Challenge 2019')
print('=' * 55)
print(f"  ARR: {metricas_m4_bpi['ARR']}%   (objetivo >70%)")
print(f"  HDA: {metricas_m4_bpi['HDA']}%   (objetivo >90%)")
print(f"  DER: {metricas_m4_bpi['DER']}%   (objetivo <10%)")
print(f"  PT:  {metricas_m4_bpi['PT']} seg/caso")
print('=' * 55)

with open('resultados/resultados_m4_bpi.json', 'w', encoding='utf-8') as f:
    json.dump({'modelo': 'M4_BPI', 'metricas': metricas_m4_bpi,
               'resultados_por_caso': resultados_m4_bpi}, f, ensure_ascii=False, indent=2)
print('\n✅ M4 BPI completado y guardado.')

Ejecutando M4 sobre 50 casos BPI Challenge 2019...

  BPI_01 [Simple  ]... → aprobado ✅ (2.14s)
  BPI_02 [Simple  ]... → aprobado ✅ (3.13s)
  BPI_03 [Simple  ]... → aprobado ✅ (2.4s)
  BPI_04 [Simple  ]... → aprobado ✅ (2.05s)
  BPI_05 [Simple  ]... → aprobado ✅ (2.26s)
  BPI_06 [Simple  ]... → aprobado ✅ (1.93s)
  BPI_07 [Simple  ]... → aprobado ✅ (1.94s)
  BPI_08 [Simple  ]... → aprobado ✅ (2.04s)
  BPI_09 [Simple  ]... → aprobado ✅ (2.32s)
  BPI_10 [Simple  ]... → aprobado ✅ (2.04s)
  BPI_11 [Simple  ]... → aprobado ✅ (2.03s)
  BPI_12 [Simple  ]... → aprobado ✅ (1.94s)
  BPI_13 [Simple  ]... → aprobado ✅ (1.61s)
  BPI_14 [Simple  ]... → aprobado ✅ (1.93s)
  BPI_15 [Simple  ]... → aprobado ✅ (2.36s)
  BPI_16 [Simple  ]... → escalado_hil 🔶 (2.84s)
  BPI_17 [Simple  ]... → aprobado ✅ (2.65s)
  BPI_18 [Simple  ]... → aprobado ✅ (1.66s)
  BPI_19 [Simple  ]... → aprobado ✅ (1.99s)
  BPI_20 [Simple  ]... → aprobado ✅ (1.94s)
  BPI_21 [Medio   ]... → aprobado ✅ (3.48s)
  BPI_22 [Medio   ]..

## 7. Comparativa final — BPI Challenge 2019 vs Dataset sintético

In [11]:
print('=' * 70)
print('  COMPARATIVA FINAL — BPI Challenge 2019')
print('  Fuente: BPI Challenge 2019 (DOI: 10.4121/uuid:d06aff4b-79f0-45e6-8ec8-e19730c248f1)')
print('=' * 70)
print(f"{'Modelo':<10} {'ARR':>8} {'HDA':>8} {'DER':>8} {'PT(s)':>8}")
print('-' * 70)

for metricas in [metricas_m1_bpi, metricas_m2_bpi, metricas_m3_bpi, metricas_m4_bpi]:
    nombre = metricas['modelo'].split('_')[0]
    print(f"{nombre:<10} {metricas['ARR']:>7}% {metricas['HDA']:>7}% {metricas['DER']:>7}% {metricas['PT']:>8}")

print('=' * 70)
print('\nDataset: 50 casos reales del BPI Challenge 2019')
print('251.734 casos totales | Purchase-to-Pay | Empresa multinacional NL')
print('Licencia: CC BY 4.0 | TU/e Eindhoven')

# Guardar comparativa
comparativa = {
    'dataset': 'BPI_Challenge_2019',
    'doi': '10.4121/uuid:d06aff4b-79f0-45e6-8ec8-e19730c248f1',
    'licencia': 'CC BY 4.0',
    'total_casos_bpi': 251734,
    'casos_evaluacion': 50,
    'resultados': {
        'M1': metricas_m1_bpi,
        'M2': metricas_m2_bpi,
        'M3': metricas_m3_bpi,
        'M4': metricas_m4_bpi
    }
}

with open('resultados/comparativa_bpi_2019.json', 'w', encoding='utf-8') as f:
    json.dump(comparativa, f, ensure_ascii=False, indent=2)

print('\n✅ Comparativa BPI guardada en resultados/comparativa_bpi_2019.json')

  COMPARATIVA FINAL — BPI Challenge 2019
  Fuente: BPI Challenge 2019 (DOI: 10.4121/uuid:d06aff4b-79f0-45e6-8ec8-e19730c248f1)
Modelo          ARR      HDA      DER    PT(s)
----------------------------------------------------------------------
M1           100.0%     0.0%    22.0%      0.0
M2            84.0%    45.5%    16.0%     0.47
M3            92.0%     9.1%    34.0%      4.1
M4            94.0%    18.2%    18.0%     2.21

Dataset: 50 casos reales del BPI Challenge 2019
251.734 casos totales | Purchase-to-Pay | Empresa multinacional NL
Licencia: CC BY 4.0 | TU/e Eindhoven

✅ Comparativa BPI guardada en resultados/comparativa_bpi_2019.json
